# 📊 Notebook 3 — FT-Transformer Training
## PaySim Transaction Fraud Detection
**Project:** Multimodal Risk Assessment in FinTech Applications | Team 30

**Runtime:** GPU or CPU OK | **Est. Time:** 15–25 min

### Steps:
1. Mount Google Drive
2. Upload PaySim CSV (or generate synthetic)
3. Feature engineering (12 features)
4. Train FT-Transformer
5. Save `ft_transformer_model.pt` and `paysim_scaler_v2.pkl`

In [ ]:
# ── STEP 1: Mount Drive ───────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
import os
SAVE_DIR = '/content/drive/MyDrive/MajorProject_Models'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'✅ Drive mounted → {SAVE_DIR}')

In [ ]:
# ── STEP 2: Install Dependencies ─────────────────────────────────────────────
!pip install -q torch scikit-learn imbalanced-learn pandas numpy matplotlib seaborn

In [ ]:
# ── STEP 3: Load PaySim Dataset ──────────────────────────────────────────────
# Option A: Upload PaySim CSV to Colab
# from google.colab import files
# uploaded = files.upload()  # Upload PS_20174392719_1491204439457_log.csv
# df = pd.read_csv(list(uploaded.keys())[0])

# Option B: Generate a synthetic replica (if you don't have the CSV)
import pandas as pd
import numpy as np

np.random.seed(42)
N_LEGIT = 50000
N_FRAUD = 2000

def gen_legit(n):
    amount = np.random.uniform(100, 10000, n)
    oldbalOrg = np.random.uniform(amount*1.2, amount*5, n)
    newbalOrg = oldbalOrg - amount
    oldbalDest = np.random.uniform(0, 50000, n)
    newbalDest = oldbalDest + amount * 0.95
    tx_type = np.random.choice([1,2,3,4,5], n, p=[0.3,0.2,0.1,0.3,0.1])
    return pd.DataFrame({'step': np.random.randint(1,743,n), 'type': tx_type,
        'amount': amount, 'oldbalanceOrg': oldbalOrg, 'newbalanceOrig': newbalOrg,
        'oldbalanceDest': oldbalDest, 'newbalanceDest': newbalDest, 'isFraud': 0})

def gen_fraud(n):
    amount = np.random.uniform(1000, 100000, n)
    oldbalOrg = amount * np.random.uniform(0.9, 1.1, n)
    newbalOrg = np.zeros(n)
    oldbalDest = np.zeros(n)
    newbalDest = amount
    tx_type = np.random.choice([2,4], n)
    return pd.DataFrame({'step': np.random.randint(1,743,n), 'type': tx_type,
        'amount': amount, 'oldbalanceOrg': oldbalOrg, 'newbalanceOrig': newbalOrg,
        'oldbalanceDest': oldbalDest, 'newbalanceDest': newbalDest, 'isFraud': 1})

df = pd.concat([gen_legit(N_LEGIT), gen_fraud(N_FRAUD)], ignore_index=True).sample(frac=1, random_state=42)
print(f'✅ Dataset: {len(df)} rows | Fraud: {df.isFraud.sum()} ({df.isFraud.mean()*100:.2f}%)')

In [ ]:
# ── STEP 4: Feature Engineering (Same 12 features as app.py) ─────────────────
import pickle
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

df['orig_diff']    = df['oldbalanceOrg'] - df['newbalanceOrig'] - df['amount']
df['dest_diff']    = df['newbalanceDest'] - df['oldbalanceDest'] - df['amount']
df['orig_zero']    = (df['newbalanceOrig'] == 0).astype(int)
df['dest_zero']    = (df['oldbalanceDest'] == 0).astype(int)
df['amount_ratio'] = df['amount'] / (df['oldbalanceOrg'] + 1)

FEATURES = ['step','type','amount','oldbalanceOrg','newbalanceOrig',
            'oldbalanceDest','newbalanceDest','orig_diff','dest_diff',
            'orig_zero','dest_zero','amount_ratio']

X = df[FEATURES].values
y = df['isFraud'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# SMOTE to balance classes
sm = SMOTE(random_state=42)
X_res, y_res = sm.fit_resample(X_train, y_train)

# Scaler
scaler = StandardScaler()
X_res_scaled  = scaler.fit_transform(X_res)
X_test_scaled = scaler.transform(X_test)

# Save scaler
with open(f'{SAVE_DIR}/paysim_scaler_v2.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print(f'✅ Features engineered | Train: {len(X_res_scaled)} (after SMOTE) | Test: {len(X_test_scaled)}')
print(f'✅ Scaler saved: {SAVE_DIR}/paysim_scaler_v2.pkl')

In [ ]:
# ── STEP 5: FT-Transformer Architecture ──────────────────────────────────────
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

class FTTransformer(nn.Module):
    def __init__(self, num_features=12, embed_dim=64, num_heads=4, num_layers=3, dropout=0.1):
        super().__init__()
        self.num_features = num_features
        self.embed_dim = embed_dim

        # Feature Tokenizer: each feature → embed_dim vector
        self.feature_projections = nn.ModuleList([
            nn.Linear(1, embed_dim) for _ in range(num_features)
        ])

        # CLS token
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))

        # Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads, dim_feedforward=embed_dim*4,
            dropout=dropout, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # Classification head
        self.classifier = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 2)
        )

    def forward(self, x):
        batch_size = x.shape[0]
        tokens = [self.feature_projections[i](x[:, i:i+1]).unsqueeze(1)
                  for i in range(self.num_features)]
        tokens = torch.cat(tokens, dim=1)

        cls = self.cls_token.expand(batch_size, -1, -1)
        tokens = torch.cat([cls, tokens], dim=1)

        out = self.transformer(tokens)
        return self.classifier(out[:, 0])

model = FTTransformer(num_features=12, embed_dim=64, num_heads=4, num_layers=3).to(device)
total = sum(p.numel() for p in model.parameters())
print(f'✅ FT-Transformer ready | Params: {total:,}')

In [ ]:
# ── STEP 6: Training ─────────────────────────────────────────────────────────
from sklearn.metrics import f1_score, roc_auc_score, classification_report
from torch.optim import AdamW

# DataLoaders
X_tr_t = torch.FloatTensor(X_res_scaled)
y_tr_t = torch.LongTensor(y_res)
X_te_t = torch.FloatTensor(X_test_scaled)
y_te_t = torch.LongTensor(y_test)

train_loader = DataLoader(TensorDataset(X_tr_t, y_tr_t), batch_size=256, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_te_t, y_te_t), batch_size=256)

criterion = nn.CrossEntropyLoss()
optimizer = AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

EPOCHS = 20
best_f1 = 0.0
history = {'loss': [], 'f1': []}

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for Xb, yb in train_loader:
        Xb, yb = Xb.to(device), yb.to(device)
        optimizer.zero_grad()
        out = model(Xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for Xb, yb in val_loader:
            preds = model(Xb.to(device)).argmax(1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(yb.numpy())

    f1 = f1_score(all_labels, all_preds, average='binary')
    avg_loss = total_loss / len(train_loader)
    history['loss'].append(avg_loss)
    history['f1'].append(f1)

    if f1 > best_f1:
        best_f1 = f1
        torch.save(model.state_dict(), f'{SAVE_DIR}/ft_transformer_model.pt')
        saved = '✅ Saved!'
    else:
        saved = ''

    if (epoch+1) % 5 == 0 or epoch == 0:
        print(f'Epoch {epoch+1:02d}/{EPOCHS} | Loss: {avg_loss:.4f} | F1: {f1:.4f} {saved}')

print(f'\n🎉 Training complete! Best F1: {best_f1:.4f}')
print(f'💾 Model saved: {SAVE_DIR}/ft_transformer_model.pt')

In [ ]:
# ── STEP 7: Final Evaluation + Plot + Download ────────────────────────────────
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

model.load_state_dict(torch.load(f'{SAVE_DIR}/ft_transformer_model.pt'))
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for Xb, yb in val_loader:
        preds = model(Xb.to(device)).argmax(1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(yb.numpy())

print('\n=== Classification Report ===')
print(classification_report(all_labels, all_preds, target_names=['LEGIT','FRAUD']))

# Plot
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(history['loss'], 'r-'); axes[0].set_title('Training Loss')
axes[1].plot(history['f1'], 'g-');   axes[1].set_title('Val F1 Score')
cm = confusion_matrix(all_labels, all_preds)
sns.heatmap(cm, annot=True, fmt='d', ax=axes[2], cmap='Blues',
            xticklabels=['LEGIT','FRAUD'], yticklabels=['LEGIT','FRAUD'])
axes[2].set_title('Confusion Matrix')
plt.suptitle('FT-Transformer — PaySim Transaction Fraud', fontsize=13)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/ft_transformer_plot.png', dpi=150)
plt.show()

# Download both files
from google.colab import files
print('\n⬇️ Downloading model files...')
files.download(f'{SAVE_DIR}/ft_transformer_model.pt')
files.download(f'{SAVE_DIR}/paysim_scaler_v2.pkl')
print('✅ Place both in: d:\\Major Project\\model\\')